In [11]:
import pandas as pd
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.multiclass import OneVsOneClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils import resample
import numpy as np

df = pd.read_csv('../winequality-red-clean.csv', delimiter=',')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1458 non-null   float64
 1   volatile acidity      1458 non-null   float64
 2   citric acid           1458 non-null   float64
 3   residual sugar        1458 non-null   float64
 4   chlorides             1458 non-null   float64
 5   free sulfur dioxide   1458 non-null   float64
 6   total sulfur dioxide  1458 non-null   float64
 7   density               1458 non-null   float64
 8   pH                    1458 non-null   float64
 9   sulphates             1458 non-null   float64
 10  alcohol               1458 non-null   float64
 11  quality               1458 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 136.8 KB


In [2]:

X = df.drop(columns=['quality'])
y = df['quality']

standard_scaler = StandardScaler()
X = standard_scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [3]:
def train_test_accuracy(train_accuracy, test_accuracy):
    print(f'Training Accuracy: {(train_accuracy * 100).__round__(3)}%')
    print(f'Testing Accuracy: {(test_accuracy * 100).__round__(3)}%')

In [14]:
def calculate_bias_variance(model, X, y, num_samples=100):
    # Array to store predictions from all bootstrapped models
    all_predictions = np.zeros((num_samples, X.shape[0]))

    # Perform bootstrapping
    for i in range(num_samples):
        # Resample the training data (bootstrap sample)
        X_resampled, y_resampled = resample(X, y)

        # Train the model on the resampled data
        model.fit(X_resampled, y_resampled)

        # Make predictions on the entire dataset (both train and test data)
        all_predictions[i, :] = model.predict(X)

    # Calculate the mean prediction for each data point
    mean_predictions = np.mean(all_predictions, axis=0)

    # Calculate the bias^2: The squared difference between true values and mean predictions
    bias_squared = np.mean((mean_predictions - y) ** 2)
    bias_percent = (bias_squared / np.mean(y_test)) * 100

    # Calculate the variance: The average variance of predictions across different bootstrapped models
    variance = np.mean(np.var(all_predictions, axis=0))
    variance_percent = (variance / np.mean(y_test)) * 100

    print(f"Bias^2 : {bias_percent:.2f}%")
    print(f"Variance: {variance_percent:.2f}%")

In [15]:
# Linear

ovr_linear = OneVsRestClassifier(SVC(kernel='linear'))
ovr_linear.fit(X_train, y_train)

cv_results = cross_validate(ovr_linear, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier LINEAR:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovr_linear, X_test, y_test)


OneVsRestClassifier LINEAR:
Training Accuracy: 56.91%
Testing Accuracy: 54.807%
Bias^2 : 6.07%
Variance: 2.28%


In [16]:
#RBF

ovr_rbf = OneVsRestClassifier(SVC(kernel='rbf'))
ovr_rbf.fit(X_train, y_train)

cv_results = cross_validate(ovr_rbf, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovr_rbf, X_test, y_test)


OneVsRestClassifier RBF:
Training Accuracy: 71.193%
Testing Accuracy: 58.369%
Bias^2 : 3.91%
Variance: 2.18%


In [17]:
# Linear

ovo_linear = OneVsOneClassifier(SVC(kernel='linear'))
ovo_linear.fit(X_train, y_train)

cv_results = cross_validate(OneVsOneClassifier(SVC(kernel='linear')), X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier LINEAR : ')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovo_linear, X_test, y_test)

OneVsOneClassifier LINEAR : 
Training Accuracy: 61.557%
Testing Accuracy: 57.751%
Bias^2 : 5.24%
Variance: 2.30%


In [18]:
# RBF
ovo_rbf = OneVsOneClassifier(SVC(kernel='rbf'))
ovo_rbf.fit(X_train, y_train)

cv_results = cross_validate(ovo_rbf, X, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovo_rbf, X_test, y_test)


OneVsOneClassifier RBF:
Training Accuracy: 69.633%
Testing Accuracy: 58.507%
Bias^2 : 4.22%
Variance: 2.10%


In [20]:
'''
Selected Features classification:
 Index(['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
       'alcohol'], dtype='object')
'''

selected_features = ['volatile acidity', 'citric acid', 'total sulfur dioxide', 'sulphates',
                     'alcohol']
X_selected = df[selected_features]
y = df['quality']

standard_scaler = StandardScaler()
X_selected = standard_scaler.fit_transform(X_selected)

X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

In [21]:
# Linear

ovr_linear = OneVsRestClassifier(SVC(kernel='linear'))
ovr_linear.fit(X_train, y_train)

cv_results = cross_validate(ovr_linear, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier LINEAR:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovr_linear, X_test, y_test)

OneVsRestClassifier LINEAR:
Training Accuracy: 54.236%
Testing Accuracy: 52.678%
Bias^2 : 7.41%
Variance: 1.45%


In [23]:
#RBF

ovr_rbf = OneVsRestClassifier(SVC(kernel='rbf'))
ovr_rbf.fit(X_train, y_train)

cv_results = cross_validate(ovr_rbf, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsRestClassifier RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovr_rbf, X_test, y_test)

OneVsRestClassifier RBF:
Training Accuracy: 65.998%
Testing Accuracy: 59.671%
Bias^2 : 5.47%
Variance: 1.78%


In [24]:
# Linear
ovo_linear = OneVsOneClassifier(SVC(kernel='linear'))
ovo_linear.fit(X_train, y_train)

cv_results = cross_validate(ovo_linear, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier LINEAR : ')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovo_linear, X_test, y_test)

OneVsOneClassifier LINEAR : 
Training Accuracy: 58.848%
Testing Accuracy: 57.618%
Bias^2 : 6.89%
Variance: 1.40%


In [25]:
# RBF
ovo_rbf = OneVsOneClassifier(SVC(kernel='rbf'))
ovo_rbf.fit(X_train, y_train)

cv_results = cross_validate(ovo_rbf, X_selected, y, cv=5, return_train_score=True)
train_accuracy = cv_results['train_score'].mean()
test_accuracy = cv_results['test_score'].mean()

print(f'OneVsOneClassifier RBF:')
train_test_accuracy(train_accuracy, test_accuracy)
calculate_bias_variance(ovo_rbf, X_test, y_test)

OneVsOneClassifier RBF:
Training Accuracy: 64.798%
Testing Accuracy: 59.397%
Bias^2 : 5.84%
Variance: 1.50%
